### Module 11 — Agentic AI Fundamentals

In [15]:
import os
import logging
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI 
from langchain_core.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

In [2]:
os.environ['ANONYMIZED_TELEMETRY'] = 'False'  
logging.getLogger('httpx').setLevel(logging.WARNING)

In [3]:
load_dotenv()

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

Weather tool

In [4]:
@tool
def get_weather(city: str) -> str:
    '''Return the current weather for a city'''
    
    fake_weather = {
        'lagos': 'Sunny, 32°C, humidity 78%',
        'abuja': 'Cloudy, 28°C, humidity 65%',
        'kano':  'Hot and dry, 36°C, humidity 30%',
    }
    
    return fake_weather.get(city.lower(), f'No weather data for {city}')

print(f'Type: {type(get_weather).__name__}')

Type: StructuredTool


Test the Tool Manually

In [5]:
print(get_weather('Lagos'))

Sunny, 32°C, humidity 78%


C:\Users\USER\AppData\Local\Temp\ipykernel_8148\1872752074.py:1: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use invoke instead.
  print(get_weather('Lagos'))


In [6]:
print('Lagos: ', get_weather('Lagos'))
print('Abuja: ', get_weather('Abuja'))
print('Kano:  ', get_weather('Kano'))
print('Paris: ', get_weather('Paris'))

Lagos:  Sunny, 32°C, humidity 78%
Abuja:  Cloudy, 28°C, humidity 65%
Kano:   Hot and dry, 36°C, humidity 30%
Paris:  No weather data for Paris


Test — Call the Tool the LangChain 

In [7]:
result = get_weather.invoke({'city': 'Lagos'})
print(f'Result: {result}')

Result: Sunny, 32°C, humidity 78%


Crop diseases tool

In [8]:
@tool
def search_crop_disease(query: str) -> str:
    '''
    Search the crop disease knowledge base for information about diseases and their treatments
    '''
    
    knowledge = {
        'cassava mosaic': 'Cassava Mosaic Disease: viral, causes yellowing and mottling of leaves. Control: use disease-free cuttings, resistant varieties, remove infected plants.',
        'maize smut':     'Maize Smut: fungal, causes grey or black galls on ears and stalks. Control: plant resistant hybrids, crop rotation, remove galls.',
        'rice blast':     'Rice Blast: fungal, causes diamond-shaped lesions on leaves. Control: resistant varieties, balanced nitrogen, fungicides.',
        'tomato leaf curl': 'Tomato Leaf Curl Virus: causes upward curling and yellowing. Control: manage whiteflies, resistant varieties, remove infected plants.',
    }
    
    query_lower = query.lower()
    
    for key, value in knowledge.items():
        if key in query_lower:
            return value
        
    return f'No knowledge found for: {query}'
    
print('search_crop_disease is now a LangChain tool')
print(f'   Type: {type(search_crop_disease).__name__}')

search_crop_disease is now a LangChain tool
   Type: StructuredTool


In [9]:
print(search_crop_disease.invoke({'query': 'cassava mosaic disease'}))

Cassava Mosaic Disease: viral, causes yellowing and mottling of leaves. Control: use disease-free cuttings, resistant varieties, remove infected plants.


In [10]:
print(search_crop_disease.invoke({'query': 'What causes tomato leaf curl?'}))

Tomato Leaf Curl Virus: causes upward curling and yellowing. Control: manage whiteflies, resistant varieties, remove infected plants.


Collect the Tools Into a List

In [12]:
tools = [get_weather, search_crop_disease]

print(f'{len(tools)} tools ready for the agent:')

for t in tools:
    print(f'    - {t.name}: {t.description[:60]}...')

2 tools ready for the agent:
    - get_weather: Return the current weather for a city...
    - search_crop_disease: Search the crop disease knowledge base for information about...


Inspect One Tool's Description

In [14]:
print('Tool name:', get_weather.name)
print()
print('Tool description:')
print(get_weather.description)
print()
print('Tool input schema:')
print(get_weather.args_schema.schema())

Tool name: get_weather

Tool description:
Return the current weather for a city

Tool input schema:
{'title': 'get_weatherSchema', 'description': 'Return the current weather for a city', 'type': 'object', 'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city']}


**AgentExecutor**
* The loop that runs the agent — think → act → observe → repeat

**create_tool_calling_agent**
* Builds the agent using OpenAI's tool-calling API

**ChatPromptTemplate**
* Builds the system prompt that tells the agent what to do

Create the Agent's Prompt

In [17]:
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Use the available tools when they help answer the user\'s question.'),
    ('human', '{input}'),
    ('placeholder', '{agent_scratchpad}'),
])

print('Agent prompt created')

Agent prompt created


Create the Agent

In [18]:
agent = create_tool_calling_agent(llm, tools, prompt)

print('Agent created')
print(f'   Type: {type(agent).__name__}')

Agent created
   Type: RunnableSequence


Wrap the Agent in the Executor

In [19]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

print('AgentExecutor ready')

AgentExecutor ready


 Visualization

In [20]:
response = agent_executor.invoke({'input': 'What is the weather in Lagos?'})



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'Lagos'}`


Sunny, 32°C, humidity 78%The weather in Lagos is sunny with a temperature of 32°C and humidity at 78%.

> Finished chain.


In [21]:
response = agent_executor.invoke({'input': 'How do I treat rice blast?'})



> Entering new AgentExecutor chain...

Invoking: `search_crop_disease` with `{'query': 'rice blast'}`


Rice Blast: fungal, causes diamond-shaped lesions on leaves. Control: resistant varieties, balanced nitrogen, fungicides.To treat rice blast, which is a fungal disease that causes diamond-shaped lesions on the leaves of rice plants, you can take the following measures:

1. **Use Resistant Varieties**: Plant rice varieties that are resistant to rice blast.

2. **Balanced Nitrogen Fertilization**: Ensure that nitrogen is applied in a balanced manner to avoid excessive growth that can make plants more susceptible to the disease.

3. **Fungicides**: Apply appropriate fungicides as a preventive measure or at the first signs of the disease.

Implementing these strategies can help manage and control rice blast effectively.

> Finished chain.


In [22]:
# Test 3 — a question that doesn't need any tool
response = agent_executor.invoke({'input': 'What is 2 + 2?'})



> Entering new AgentExecutor chain...
2 + 2 equals 4.

> Finished chain.


In [23]:
# Test 4 — a question that combines both tools
response = agent_executor.invoke({'input': 'What is the weather in Kano, and how do I treat maize smut?'})



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'Kano'}`


Hot and dry, 36°C, humidity 30%
Invoking: `search_crop_disease` with `{'query': 'maize smut'}`


Maize Smut: fungal, causes grey or black galls on ears and stalks. Control: plant resistant hybrids, crop rotation, remove galls.The current weather in Kano is hot and dry, with a temperature of 36°C and humidity at 30%.

Regarding maize smut, it is a fungal disease that causes grey or black galls on the ears and stalks of maize plants. Here are some treatment and control measures:

1. **Plant Resistant Hybrids**: Use maize varieties that are resistant to smut.
2. **Crop Rotation**: Rotate crops to reduce the incidence of the disease.
3. **Remove Galls**: Regularly remove and destroy any galls that appear on the plants to prevent the spread of the fungus.

Implementing these strategies can help manage and control maize smut effectively.

> Finished chain.


In [24]:
# Test 5 — a trick question
response = agent_executor.invoke({'input': 'What is the weather on Mars?'})



> Entering new AgentExecutor chain...
I can only provide weather information for cities on Earth. If you have a specific city in mind, please let me know!

> Finished chain.
